In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.family'] = 'Malgun Gothic'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 데이터 전처리 관련 ################################################
# 결측치 처리
from sklearn.impute import SimpleImputer
# 표준화
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
# 인코더
from sklearn.preprocessing import LabelEncoder
# 원핫 인코더
from sklearn.preprocessing import OneHotEncoder

# 학습 모델 성능 관련 #######################################
# 원하는 비율로 데이터를 나누기 위해
from sklearn.model_selection import train_test_split
# K-Fold 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
# 학습 곡선
from sklearn.model_selection import learning_curve
# 하이퍼 파라미터 튜닝
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 모델 성능 평가 ########################
# 회귀용
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score
# 분류용
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_curve
from sklearn.metrics import auc
from sklearn.metrics import roc_auc_score
from sklearn.metrics import ConfusionMatrixDisplay

# 피처 선택 #########################
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import RFE
from sklearn.inspection import permutation_importance

# 학습 모델 ########################
# 분류
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier 
from sklearn.naive_bayes import GaussianNB
from catboost import CatBoostClassifier

# 회귀
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import BayesianRidge
from catboost import CatBoostRegressor

# 결정트리를 시각화할 수 있는 라이브러리
from sklearn.tree import plot_tree

# 차원 축소
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.manifold import TSNE

# 연관 규칙 학습
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import fpgrowth
from mlxtend.frequent_patterns import association_rules

# 군집
from sklearn.cluster import KMeans
from sklearn.cluster import DBSCAN
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.mixture import GaussianMixture
from sklearn.cluster import MeanShift, estimate_bandwidth

# 파이프라인
from sklearn.pipeline import Pipeline

# KDE를 그리기 위한 통계값을 구할 수 있는 함수
from scipy.stats import gaussian_kde

# 피어슨 상관 계수 (연속형 수치형 데이터 vs 연속형 수치형 데이터)
from scipy.stats import pearsonr
# 카이제곱 검증 (범주형 데이터 vs 범주형 데이터, 순위X)
from scipy.stats import chi2_contingency
# 스피어만 상관계수 (범주형 데이터 vs 범주형 데이터, 순위O)
from scipy.stats import spearmanr
# 포인트 이분 상관계수 (범주형 데이터 vs 연속형 수치형 데이터)
from scipy.stats import pointbiserialr

# 오버 샘플링
from imblearn.over_sampling import SMOTE

# 객체를 파일에 저장
import pickle

# 문장 데이터 관련
# 단어 토큰화
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
# 정규식 사용
import re

# 불필요한 경고 뜨지 않게
import warnings
warnings.filterwarnings('ignore')

### 예시 문자열로 테스트

In [3]:
raw_sms = "	<div>FREE!!</div> Win a ε1000 cash prize! Go to http://spam.com or call 0800-123-456.	"

In [15]:
# 노이즈를 제거하는 함수
def clean_noise_expert(text) :
    print(f'원본 데이터 : {text}')
    # HTML 태그 제거
    # < > 사이의 모든 문자를 찾아 제거한다.
    text = re.sub(r'<.*?>', ' ', text) 
    print(f'태그 제거 : {text}')

    # 특수문자 제거 (알파벳과 공백만 유지)
    # 알파벳 (a-z, A-Z)와 공백( ) 이 아닌 모든 문자를 공백으로 바꾼다.
    text = re.sub(r'[^a-zA-Z ]', ' ', text)
    print(f'특수문자 제거 : {text}')

    # 숫자 제거 (필요한 경우)
    # \d+ 패턴은 하나 이상의 연속된 숫자를 의미한다.
    text = re.sub(r'\d+', ' ', text)
    print(f'숫자 제거 : {text}')

    # 다중 공백 및 양끝 공백 제거
    # \s+는 모든 공백 문자(탭, 줄바꿈 포함)를 의미하며, 이를 단일 공백으로 변경한다.
    text = re.sub(r'\s+', ' ', text).strip()
    print(f'공백 변경 : {text}')
    return text

In [16]:
final_result = clean_noise_expert(raw_sms)

원본 데이터 : 	<div>FREE!!</div> Win a ε1000 cash prize! Go to http://spam.com or call 0800-123-456.	
태그 제거 : 	 FREE!!  Win a ε1000 cash prize! Go to http://spam.com or call 0800-123-456.	
특수문자 제거 :   FREE    Win a       cash prize  Go to http   spam com or call               
숫자 제거 :   FREE    Win a       cash prize  Go to http   spam com or call               
공백 변경 : FREE Win a cash prize Go to http spam com or call


### 스팸 문자 데이터에 적용

In [17]:
df = pd.read_csv('data/spam3.csv', encoding='latin-1')
df

,label,text,tokenized_text
0,ham,"Go until jurong point, crazy.. Available only ...","['Go', 'until', 'jurong', 'point', ',', 'crazy..."
1,ham,Ok lar... Joking wif u oni...,"['Ok', 'lar', '...', 'Joking', 'wif', 'u', 'on..."
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,"['Free', 'entry', 'in', '2', 'a', 'wkly', 'com..."
3,ham,U dun say so early hor... U c already then say...,"['U', 'dun', 'say', 'so', 'early', 'hor', '......"
4,ham,"Nah I don't think he goes to usf, he lives aro...","['Nah', 'I', 'do', ""n't"", 'think', 'he', 'goes..."
...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,"['This', 'is', 'the', '2nd', 'time', 'we', 'ha..."
5568,ham,Will Ì_ b going to esplanade fr home?,"['Will', 'Ì_', 'b', 'going', 'to', 'esplanade'..."
5569,ham,"Pity, * was in mood for that. So...any other s...","['Pity', ',', '*', 'was', 'in', 'mood', 'for',..."
5570,ham,The guy did some bitching but I acted like i'd...,"['The', 'guy', 'did', 'some', 'bitching', 'but..."


In [18]:
# 노이즈 제거 함수
def clean_noise(text) :
    # HTML 태그 제거
    text = re.sub(r'<.*?>', ' ', text)
    # 알파벳과 공백이 아닌 것들 제거
    text = re.sub(r'[^a-zA-Z ]', ' ', text)
    # 숫자 제거 : 숫자가 분석에 큰 의미가 없다면 공백으로 치환
    text = re.sub(r'\d+', ' ', text)
    # 연속된 공백을 하나로 변경
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [19]:
# 처리한다.
df['clean_text'] = df['text'].apply(clean_noise)
df

,label,text,tokenized_text,clean_text
0,ham,"Go until jurong point, crazy.. Available only ...","['Go', 'until', 'jurong', 'point', ',', 'crazy...",Go until jurong point crazy Available only in ...
1,ham,Ok lar... Joking wif u oni...,"['Ok', 'lar', '...', 'Joking', 'wif', 'u', 'on...",Ok lar Joking wif u oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,"['Free', 'entry', 'in', '2', 'a', 'wkly', 'com...",Free entry in a wkly comp to win FA Cup final ...
3,ham,U dun say so early hor... U c already then say...,"['U', 'dun', 'say', 'so', 'early', 'hor', '......",U dun say so early hor U c already then say
4,ham,"Nah I don't think he goes to usf, he lives aro...","['Nah', 'I', 'do', ""n't"", 'think', 'he', 'goes...",Nah I don t think he goes to usf he lives arou...
...,...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,"['This', 'is', 'the', '2nd', 'time', 'we', 'ha...",This is the nd time we have tried contact u U ...
5568,ham,Will Ì_ b going to esplanade fr home?,"['Will', 'Ì_', 'b', 'going', 'to', 'esplanade'...",Will b going to esplanade fr home
5569,ham,"Pity, * was in mood for that. So...any other s...","['Pity', ',', '*', 'was', 'in', 'mood', 'for',...",Pity was in mood for that So any other suggest...
5570,ham,The guy did some bitching but I acted like i'd...,"['The', 'guy', 'did', 'some', 'bitching', 'but...",The guy did some bitching but I acted like i d...


In [20]:
df.to_csv('data/spam4.csv', encoding='latin-1', index=False)